### **Semantic / Embedding-Aware Chunking*

In [2]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, util
from nltk.tokenize import sent_tokenize
import nltk

In [3]:
## - Step 1: Fetch PDF

reader = PdfReader("./docs/PAPER1.pdf")

text = ""
for page in reader.pages:
    text += page.extract_text()

print("Text length:", len(text))

Text length: 39587


In [4]:
## - Step 2: Sentence Split

sentences = sent_tokenize(text)

print("Total sentences:", len(sentences))

Total sentences: 359


In [5]:
## - Step 3: Load embedding model

model = SentenceTransformer("all-MiniLM-L6-v2")

sentence_embeddings = model.encode(sentences, convert_to_tensor=True)

sentence_embeddings[0][:10]

tensor([-0.0184,  0.0098, -0.0044,  0.0081,  0.0201, -0.0086, -0.0528, -0.0043,
        -0.0433,  0.1206])

In [6]:

## - Step 4: Semantic Chunking


threshold = 0.7
semantic_chunks = []
current_chunk = [sentences[0]]

for i in range(1, len(sentences)):

    sim = util.cos_sim(
        sentence_embeddings[i-1],
        sentence_embeddings[i]
    ).item()

    if sim > threshold:
        current_chunk.append(sentences[i])
    else:
        semantic_chunks.append(" ".join(current_chunk))
        current_chunk = [sentences[i]]

if current_chunk:
    semantic_chunks.append(" ".join(current_chunk))

print("Total chunks:", len(semantic_chunks))

Total chunks: 351


In [7]:
## - Step 5: Create chunk embeddings


chunk_embeddings = model.encode(semantic_chunks)

print("Embedding shape:", chunk_embeddings.shape)

Embedding shape: (351, 384)


In [14]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# create LangChain embedding wrapper
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# pair text with embeddings
text_embedding_pairs = list(zip(semantic_chunks, chunk_embeddings))

# create FAISS vector store
faiss_db = FAISS.from_embeddings(
    text_embeddings=text_embedding_pairs,
    embedding=embedding_model
)

print("FAISS DB created successfully")

FAISS DB created successfully


In [13]:
query = "what is multi-head attention mechanism?"

results = faiss_db.similarity_search(query, k=3)

for r in results:
    print(r.page_content)

(right) Multi-Head Attention consists of several
attention layers running in parallel.
Multi-head attention allows the model to jointly attend to information from different representation
subspaces at different positions.
The first is a multi-head self-attention mechanism, and the second is a simple, position-
wise fully connected feed-forward network.
